In [3]:
import pdfplumber
import pandas as pd

def extract_ausnet_data(pdf_path):
    all_data = {}
    
    with pdfplumber.open(pdf_path) as pdf:
        page_4 = pdf.pages[3]
        table_4 = page_4.extract_table({
            "vertical_strategy": "lines",
            "horizontal_strategy": "lines",
        })
        
        if table_4:
            df_usage = pd.DataFrame(table_4)
            df_usage.columns = df_usage.iloc[0]
            df_usage = df_usage.drop(0).reset_index(drop=True)
            all_data['usage_forecast'] = df_usage
            print("Successfully extracted Usage Forecast Table (Page 4).")

        page_5 = pdf.pages[4]
        table_5 = page_5.extract_table()
        
        if table_5:
            df_costs = pd.DataFrame(table_5[1:], columns=table_5[0])
            all_data['acquisition_costs'] = df_costs
            print("Successfully extracted Acquisition Costs Table (Page 5).")

        page_3 = pdf.pages[2]
        assumptions_text = page_3.extract_text()
        all_data['assumptions_summary'] = assumptions_text

    return all_data

pdf_file = "../data/raws/energy.pdf"
try:
    results = extract_ausnet_data(pdf_file)
    
    # Lưu kết quả ra file Excel với nhiều sheet để tiện phân tích
    with pd.ExcelWriter("AusNet_Analysis_Output.xlsx") as writer:
        if 'usage_forecast' in results:
            results['usage_forecast'].to_excel(writer, sheet_name='Usage_MWh', index=False)
        if 'acquisition_costs' in results:
            results['acquisition_costs'].to_excel(writer, sheet_name='Equipment_Costs', index=False)
            
    print("\n--- HOÀN THÀNH ---")
    print("Dữ liệu đã được xuất ra file: AusNet_Analysis_Output.xlsx")

except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file '{pdf_file}'. Hãy đảm bảo file nằm cùng thư mục với script.")
except Exception as e:
    print(f"Đã xảy ra lỗi: {e}")

Successfully extracted Usage Forecast Table (Page 4).
Successfully extracted Acquisition Costs Table (Page 5).

--- HOÀN THÀNH ---
Dữ liệu đã được xuất ra file: AusNet_Analysis_Output.xlsx


In [7]:
import pdfplumber
import os

def table_to_markdown(table):
    """Chuyển đổi list of lists thành bảng Markdown sạch."""
    if not table: 
        return ""
    
    # Loại bỏ các dòng mà tất cả các ô đều rỗng (lỗi PDF thường gặp)
    table = [row for row in table if any(cell and str(cell).strip() for cell in row)]
    if not table: 
        return ""
    
    # Làm sạch dữ liệu: thay xuống dòng bằng dấu cách, rỗng thành N/A
    clean_table = [[str(cell).replace("\n", " ").strip() if cell else "N/A" for cell in row] for row in table]
    
    headers = clean_table[0]
    md_table = "| " + " | ".join(headers) + " |\n"
    md_table += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    
    for row in clean_table[1:]:
        md_table += "| " + " | ".join(row) + " |\n"
    return md_table + "\n"

def pdf_to_markdown_for_rag(pdf_path, output_path):
    if not os.path.exists(pdf_path):
        print(f"Lỗi: Không tìm thấy file tại {pdf_path}")
        return

    with pdfplumber.open(pdf_path) as pdf:
        with open(output_path, "w", encoding="utf-8") as f:
            # Metadata ở đầu file giúp Vector DB hoạt động hiệu quả hơn
            f.write("# Document: Household Energy Cost Analysis\n")
            f.write("# Source: AusNet\n\n")

            for i, page in enumerate(pdf.pages):
                f.write(f"## Page {i+1}\n\n")
                
                # 1. Trích xuất văn bản
                text = page.extract_text()
                if text:
                    # Làm sạch khoảng trắng thừa trong text
                    clean_text = "\n".join([line.strip() for line in text.split("\n") if line.strip()])
                    f.write(clean_text + "\n\n")
                
                # 2. Trích xuất bảng bằng hàm table_to_markdown đã tối ưu
                tables = page.extract_tables()
                for table in tables:
                    f.write(table_to_markdown(table))
                
                f.write("---\n\n")

# Thực thi - Đảm bảo đường dẫn đúng với cấu trúc folder của bạn
pdf_path = "../data/raws/energy.pdf"
output_path = "../data/outputs/energy_for_rag.md"

pdf_to_markdown_for_rag(pdf_path, output_path)
print(f"Đã tạo file {output_path} thành công!")

Đã tạo file ../data/outputs/energy_for_rag.md thành công!
